[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/15.%20The%20Vessel%20Re-stow%20Problem%20%28Transshipment%29/P15-Tier-1_executed.ipynb)

---


# 15. The Vessel Re-stow Problem (Transshipment)
## Tier 1: The Pen-and-Paper Method (Mixed-Integer Programming Formulation)

### Purpose and scope

This notebook formulates and solves a small vessel re-stow assignment problem as a mixed-integer program (MIP). A container is an item to be placed, and a vessel position is a possible location for it. The inputs are the containers, their current positions, and the available vessel positions. The model assigns each container to one position while minimizing the handling cost of moving containers. It also enforces position accessibility, individual position weight limits, and a simplified weight limit for each bay. The outputs are an optimal assignment for this model, its movement and cost totals, and visual summaries of the assignment.

A feasible solution is an assignment that satisfies every constraint. The objective function gives each feasible assignment a cost, and the solver searches for a feasible assignment with the smallest objective value. The model uses binary decision variables, which can take only the values 0 and 1, to represent assignment and movement decisions. For example, an assignment variable equal to 1 means that a particular container occupies a particular position; a value of 0 means that it does not. A MIP is an optimization model in which some or all decision variables are required to take integer values; here, all decision variables are binary.

The example contains six containers and six positions arranged in three bays, one row, and two tiers. Containers are not assigned new discharge-port segregation requirements: no such constraint is part of the mathematical model. In particular, the model does not require containers for one port to occupy bays separate from containers for another port. The weight limit for a bay is a simplified stability proxy, not a complete vessel-stability calculation. Stacking, lashing, crane sequencing, and other operational constraints are outside this model.

### Key modeling assumptions

- Every container must be assigned to exactly one available position.
- Every position can hold at most one container.
- Inaccessible positions cannot be assigned.
- A container can occupy a position only if its weight does not exceed that position's weight limit.
- The total container weight assigned to each bay cannot exceed the model's bay limit of 100 tons.
- Each container assigned to a position different from its current position incurs one handling cost. A container that stays in its current position incurs no move cost.

### Solution approach

1. Define binary assignment variables for container-position pairs and binary indicators for container moves.
2. Minimize total handling cost.
3. Add assignment, position-capacity, accessibility, weight, and movement constraints.
4. Solve the MIP with PuLP's CBC solver and check the solver status.
5. Extract the assignment, verify key properties, and interpret the resulting plan.

The data in the code define the concrete example used throughout the notebook. In the starting layout, each bay contains one HKG container and one SHA container, in different tiers. The optimization model is free to retain these positions because it has no port-segregation constraint; it changes a position only when doing so can improve the stated objective while satisfying the modeled constraints.

## Mathematical formulation

The model must make two linked decisions: where each container is placed, and whether that placement counts as a move. Intuitively, an assignment variable records a yes-or-no placement choice, while a move indicator records whether the chosen position differs from the container's starting position. Both are needed: the assignment variables describe the plan, and the move indicators let the objective charge for moved containers. The MIP solver searches for values of these variables that satisfy every constraint and minimize the objective.

Let $C$ be the set of containers, $P$ the set of available positions, and $B$ the set of bays. For each container $c \in C$, let $w_c$ be its weight and $q_c$ its current position. For each position $p \in P$, let $b(p)$ be its bay, $a_p \in \{0,1\}$ indicate whether it is accessible, and $u_p$ be its weight limit. Let $U_b$ be the weight limit for bay $b$, and let $h$ be the handling cost per moved container. In this example, the code imposes $U_b=100$ tons for every bay. Container weights and handling costs are nonnegative in the stated model.

Define $x_{cp}=1$ if container $c$ is assigned to position $p$, and $0$ otherwise. Define $m_c=1$ if container $c$ moves, and $0$ otherwise. These are binary decision variables: $x_{cp},m_c\in\{0,1\}$. Since each move has the same handling cost $h$, the objective is to minimize the total cost:

$$\min \quad h\sum_{c\in C}m_c$$

The summation counts moved containers, and multiplying by $h$ converts that count into total handling cost. The first constraint below assigns every container to exactly one position. The second allows at most one container in each position. The third rules out assignments to inaccessible positions:

$$\sum_{p\in P}x_{cp}=1 \quad (c\in C), \qquad
\sum_{c\in C}x_{cp}\leq 1 \quad (p\in P), \qquad
x_{cp}\leq a_p \quad (c\in C,\ p\in P).$$

A container-position assignment is allowed only when that container fits the position's weight limit. In addition, the total weight assigned within a bay must not exceed the bay limit:

$$w_c x_{cp}\leq u_p \quad (c\in C,\ p\in P), \qquad
\sum_{c\in C}\sum_{p\in P:\ b(p)=b}w_c x_{cp}\leq U_b \quad (b\in B).$$

Because $x_{cp}$ is binary and container weights are nonnegative, the individual position constraint permits the assignment when $x_{cp}=0$ and requires $w_c\leq u_p$ when $x_{cp}=1$. The code implements this same rule by fixing $x_{cp}=0$ for any pair that exceeds the position's weight limit. The bay constraint sums the weights of all containers assigned to positions in that bay; the code uses a 100-ton limit for each bay.

To connect the move indicator to the assignment, let $P_c^0=\{p\in P:p=q_c\}$ be the set of positions whose coordinates match container $c$'s current position. A container assigned to any position outside $P_c^0$ must move. The constraints below enforce this relationship in both directions:

$$m_c\geq x_{cp} \quad (c\in C,\ p\in P\setminus P_c^0), \qquad
m_c\leq\sum_{p\in P\setminus P_c^0}x_{cp} \quad (c\in C).$$

Together with the one-position assignment constraint, these inequalities make $m_c$ equal to 1 exactly when the assigned position differs from the current position. If the current position is not available, every feasible assignment counts as a move. The code compares the full $(\text{bay},\text{row},\text{tier})$ coordinate tuple, so a position is considered unchanged only when all three coordinates match.

In the code, `x[(c_idx, p_idx)]` represents $x_{cp}$, `moves[c_idx]` represents $m_c$, and `handling_cost_per_move` represents $h$. The constraints are added inside `solve_mip_restow`. The field `penalty_per_extra_move` is retained in the problem data but is not part of this objective: the model defines a cost per moved container and does not define a separate category of extra moves. The formulation includes no discharge-port segregation constraint.

The model has $|C||P|+|C|$ binary variables: one assignment variable for each container-position pair and one move indicator for each container. A MIP solver searches over feasible integer assignments; worst-case solution effort can grow exponentially with the number of binary variables, although actual performance depends on the instance and solver. CBC reports whether it found an optimal solution, an infeasible model, or another termination status.

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple
import math

import matplotlib.pyplot as plt
import pandas as pd
import pulp
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """Describe a container, its destination, weight, and current vessel position."""

    id: str
    destination_port: str
    weight: float  # tons
    current_position: Tuple[int, int, int]  # (bay, row, tier)


@dataclass
class Position:
    """Describe a vessel position and the limits relevant to this model."""

    bay: int
    row: int
    tier: int
    accessible: bool = True
    max_weight: float = 50.0  # tons per position


@dataclass
class VesselRestowProblem:
    """Collect the data and cost parameters for the vessel re-stow model."""

    containers: List[Container]
    positions: List[Position]
    handling_cost_per_move: float = 150.0  # dollars per move
    penalty_per_extra_move: float = 50.0  # retained data field; not used by this formulation

In [ ]:
def create_example_problem():
    """Create the six-container, six-position example used in this notebook."""

    containers = [
        Container("HKG-1", "HKG", 20.0, (1, 1, 1)),
        Container("SHA-1", "SHA", 18.0, (1, 1, 2)),
        Container("HKG-2", "HKG", 22.0, (2, 1, 1)),
        Container("SHA-2", "SHA", 19.0, (2, 1, 2)),
        Container("HKG-3", "HKG", 21.0, (3, 1, 1)),
        Container("SHA-3", "SHA", 20.0, (3, 1, 2)),
    ]

    positions = []
    for bay in [1, 2, 3]:
        for row in [1]:
            for tier in [1, 2]:
                positions.append(Position(bay, row, tier))

    return VesselRestowProblem(containers, positions)


problem = create_example_problem()
print(
    f"Problem created with {len(problem.containers)} containers "
    f"and {len(problem.positions)} positions."
)

In [ ]:
def solve_mip_restow(problem: VesselRestowProblem):
    """Build and solve the mixed-integer vessel re-stow assignment model."""

    model = pulp.LpProblem("Vessel_Restow_Optimization", pulp.LpMinimize)

    # Binary assignment variables encode the yes-or-no placement choices of the MIP.
    x = {}
    for c_idx, container in enumerate(problem.containers):
        for p_idx, position in enumerate(problem.positions):
            x[(c_idx, p_idx)] = pulp.LpVariable(
                f"x_{c_idx}_{p_idx}",
                lowBound=0,
                upBound=1,
                cat=pulp.LpBinary,
            )

    # A separate binary indicator makes each container's movement explicit in the objective.
    moves = {}
    for c_idx, container in enumerate(problem.containers):
        moves[c_idx] = pulp.LpVariable(
            f"move_{c_idx}",
            lowBound=0,
            upBound=1,
            cat=pulp.LpBinary,
        )

    model += pulp.lpSum(
        problem.handling_cost_per_move * moves[c_idx]
        for c_idx in range(len(problem.containers))
    ), "Total_Handling_Cost"

    # Assignment and capacity constraints define a one-to-one placement of containers.
    for c_idx in range(len(problem.containers)):
        model += (
            pulp.lpSum(
                x[(c_idx, p_idx)] for p_idx in range(len(problem.positions))
            )
            == 1,
            f"Assign_Container_{c_idx}",
        )

    for p_idx in range(len(problem.positions)):
        model += (
            pulp.lpSum(
                x[(c_idx, p_idx)] for c_idx in range(len(problem.containers))
            )
            <= 1,
            f"Position_Capacity_{p_idx}",
        )

    # Fixing forbidden assignment variables to zero implements accessibility and weight limits.
    for c_idx, container in enumerate(problem.containers):
        for p_idx, position in enumerate(problem.positions):
            if not position.accessible:
                model += x[(c_idx, p_idx)] == 0, f"Inaccessible_{c_idx}_{p_idx}"
            if container.weight > position.max_weight:
                model += x[(c_idx, p_idx)] == 0, f"Position_Weight_{c_idx}_{p_idx}"

    # These inequalities make the move indicator match the chosen position exactly.
    for c_idx, container in enumerate(problem.containers):
        current_position = container.current_position
        noncurrent_position_indices = []

        for p_idx, position in enumerate(problem.positions):
            coordinates = (position.bay, position.row, position.tier)
            if coordinates == current_position:
                continue

            noncurrent_position_indices.append(p_idx)
            model += (
                moves[c_idx] >= x[(c_idx, p_idx)],
                f"Move_Required_{c_idx}_{p_idx}",
            )

        model += (
            moves[c_idx]
            <= pulp.lpSum(
                x[(c_idx, p_idx)] for p_idx in noncurrent_position_indices
            ),
            f"Move_Indicator_Upper_Bound_{c_idx}",
        )

    # The bay limit is a simplified stability proxy in this formulation.
    bays = sorted({position.bay for position in problem.positions})
    for bay in bays:
        bay_weight = pulp.lpSum(
            problem.containers[c_idx].weight * x[(c_idx, p_idx)]
            for c_idx in range(len(problem.containers))
            for p_idx, position in enumerate(problem.positions)
            if position.bay == bay
        )
        model += bay_weight <= 100, f"Weight_Limit_Bay_{bay}"

    print("Solving the MIP model...")
    model.solve(pulp.PULP_CBC_CMD(msg=False))

    return model, x, moves

In [ ]:
model, x_vars, move_vars = solve_mip_restow(problem)

status = pulp.LpStatus[model.status]
print(f"Solution status: {status}")

if status != "Optimal":
    raise RuntimeError(f"The MIP solver did not report an optimal solution: {status}")

total_cost = pulp.value(model.objective)
if total_cost is None:
    raise RuntimeError("The solver returned no objective value.")

assignments = []
total_moves = 0

for c_idx, container in enumerate(problem.containers):
    selected_positions = [
        p_idx
        for p_idx, position in enumerate(problem.positions)
        if pulp.value(x_vars[(c_idx, p_idx)]) > 0.5
    ]
    if len(selected_positions) != 1:
        raise AssertionError("Each container must have exactly one selected position.")

    p_idx = selected_positions[0]
    position = problem.positions[p_idx]
    is_move = pulp.value(move_vars[c_idx]) > 0.5
    destination_position = (position.bay, position.row, position.tier)
    actual_move = destination_position != container.current_position

    if is_move != actual_move:
        raise AssertionError("The move indicator does not match the assignment.")
    if not position.accessible:
        raise AssertionError("The assignment uses an inaccessible position.")
    if container.weight > position.max_weight:
        raise AssertionError("The assignment exceeds a position weight limit.")

    assignments.append(
        {
            "container": container.id,
            "destination": container.destination_port,
            "from_position": container.current_position,
            "to_position": destination_position,
            "is_move": is_move,
            "weight": container.weight,
        }
    )
    total_moves += int(is_move)

assigned_positions = [assignment["to_position"] for assignment in assignments]
if len(assigned_positions) != len(set(assigned_positions)):
    raise AssertionError("A vessel position has been assigned to multiple containers.")

bay_weights = {}
for assignment in assignments:
    bay = assignment["to_position"][0]
    bay_weights[bay] = bay_weights.get(bay, 0.0) + assignment["weight"]
if any(weight > 100 for weight in bay_weights.values()):
    raise AssertionError("A bay exceeds the model's 100-ton weight limit.")

if not math.isclose(
    total_cost,
    problem.handling_cost_per_move * total_moves,
    rel_tol=0.0,
    abs_tol=1e-6,
):
    raise AssertionError("The objective value does not match the move count and unit cost.")

print(f"Total handling cost: ${total_cost:.2f}")
print(f"Total container moves: {total_moves}")

In [ ]:
assignment_df = pd.DataFrame(assignments)
print("\nDetailed container assignments:")
print(
    assignment_df[
        ["container", "destination", "from_position", "to_position", "is_move"]
    ].to_string(index=False)
)

print("\nContainer positions grouped by destination port:")
for port in assignment_df["destination"].unique():
    port_containers = assignment_df[assignment_df["destination"] == port]
    print(f"\n{port} containers:")
    for _, row in port_containers.iterrows():
        move_status = "moved" if row["is_move"] else "not moved"
        bay, row_number, tier = row["to_position"]
        print(
            f"  {row['container']}: bay {bay}, row {row_number}, "
            f"tier {tier} ({move_status})"
        )

In [ ]:
def visualize_restow_solution(assignments, problem, title="Vessel Re-stow Solution"):
    """Plot current and assigned positions, with color indicating destination port."""

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].set_title("Current configuration", fontweight="bold")
    axes[1].set_title("Assigned configuration", fontweight="bold")

    for ax in axes:
        ax.set_xlabel("Bay")
        ax.set_ylabel("Tier")

    port_colors = {"HKG": "red", "SHA": "blue"}
    for container in problem.containers:
        bay, row, tier = container.current_position
        color = port_colors.get(container.destination_port, "gray")
        axes[0].scatter(bay, tier, s=300, c=color, alpha=0.7, edgecolors="black")
        axes[0].text(
            bay, tier, container.id, ha="center", va="center",
            fontweight="bold", fontsize=8
        )

    for assignment in assignments:
        bay, row, tier = assignment["to_position"]
        color = port_colors.get(assignment["destination"], "gray")
        alpha = 0.35 if assignment["is_move"] else 0.7
        axes[1].scatter(bay, tier, s=300, c=color, alpha=alpha, edgecolors="black")
        axes[1].text(
            bay, tier, assignment["container"], ha="center", va="center",
            fontweight="bold", fontsize=8
        )

    ports = sorted({container.destination_port for container in problem.containers})
    legend_handles = [
        plt.Line2D(
            [0], [0], marker="o", color="w",
            markerfacecolor=port_colors.get(port, "gray"),
            markersize=10, label=port,
        )
        for port in ports
    ]

    bays = sorted({position.bay for position in problem.positions})
    tiers = sorted({position.tier for position in problem.positions})
    for ax in axes:
        ax.legend(handles=legend_handles, title="Destination port", loc="upper right")
        ax.set_xlim(min(bays) - 0.5, max(bays) + 0.5)
        ax.set_ylim(min(tiers) - 0.5, max(tiers) + 0.5)
        ax.set_xticks(bays)
        ax.set_yticks(tiers)
        ax.grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()


visualize_restow_solution(assignments, problem)

In [ ]:
def analyze_solution_performance(assignments, total_cost, problem):
    """Summarize movement, cost, bay use, and destination-port distribution."""

    moved_containers = [assignment for assignment in assignments if assignment["is_move"]]
    total_weight_moved = sum(assignment["weight"] for assignment in moved_containers)

    bay_ports = {}
    bay_counts = {}
    for assignment in assignments:
        bay = assignment["to_position"][0]
        bay_counts[bay] = bay_counts.get(bay, 0) + 1
        bay_ports.setdefault(bay, set()).add(assignment["destination"])

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    cost_ax = axes[0, 0]
    cost_ax.bar(
        ["Total handling cost"], [total_cost],
        color="skyblue", label="Handling cost",
    )
    cost_ax.set_title("Handling cost", fontweight="bold")
    cost_ax.set_ylabel("Cost (dollars)")
    cost_ax.legend()
    cost_ax.text(0, total_cost, f"${total_cost:.2f}", ha="center", va="bottom")

    movement_ax = axes[0, 1]
    movement_counts = [len(moved_containers), len(assignments) - len(moved_containers)]
    movement_labels = ["Moved", "Not moved"]
    if sum(movement_counts) > 0:
        movement_ax.pie(
            movement_counts, labels=movement_labels,
            colors=["orange", "lightgreen"], autopct="%1.0f%%", startangle=90,
        )
    movement_ax.set_title("Container movement", fontweight="bold")

    utilization_ax = axes[1, 0]
    bays = sorted({position.bay for position in problem.positions})
    utilization_ax.bar(bays, [bay_counts.get(bay, 0) for bay in bays], color="teal")
    utilization_ax.set_title("Container distribution by bay", fontweight="bold")
    utilization_ax.set_xlabel("Bay number")
    utilization_ax.set_ylabel("Number of containers")
    utilization_ax.set_xticks(bays)

    port_ax = axes[1, 1]
    ports = sorted({container.destination_port for container in problem.containers})
    port_bay_counts = [
        sum(port in destinations for destinations in bay_ports.values())
        for port in ports
    ]
    port_ax.bar(ports, port_bay_counts, color=["red", "blue"][:len(ports)])
    port_ax.set_title("Bays used by each destination port", fontweight="bold")
    port_ax.set_xlabel("Destination port")
    port_ax.set_ylabel("Number of bays")

    fig.suptitle("MIP solution performance", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

    print("\nSolution performance summary")
    print(f"Total containers: {len(assignments)}")
    print(f"Containers moved: {len(moved_containers)}")
    print(f"Total weight moved: {total_weight_moved:.1f} tons")
    print(f"Total handling cost: ${total_cost:.2f}")
    if moved_containers:
        print(f"Handling cost per moved container: ${total_cost / len(moved_containers):.2f}")
    else:
        print("Handling cost per moved container: not applicable (no containers moved)")
    for port, bay_count in zip(ports, port_bay_counts):
        print(f"{port} containers occupy {bay_count} bay(s)")


analyze_solution_performance(assignments, total_cost, problem)

## Interpreting the solution

The objective value is the sum of the handling costs for containers whose assigned coordinates differ from their current coordinates. The assignment table identifies each container's origin and destination position; the move indicator makes the objective's meaning visible at the container level. The visualizations summarize the same assignment and should be read alongside the model's stated scope. The position plot displays bay and tier; the example has a single row, so row is not a plotted dimension. In the assigned-configuration plot, lighter markers indicate moved containers, while color identifies destination port.

An optimal solver status means that the returned feasible assignment has the minimum objective value for this formulated MIP, subject to the solver's optimality test. It does not establish feasibility or optimality for operational requirements that the formulation omits, such as port segregation, stacking rules, or crane sequencing. Multiple assignments can have the same optimal cost; therefore, the solver may return any one of them.

### Strengths and limitations of the MIP approach

The formulation makes assignment decisions and modeled constraints explicit, and a solver can certify optimality for the mathematical model when it reports an optimal status. Its practical computational effort can increase substantially as the number of binary assignment variables grows. The quality and operational relevance of the result also depend on whether the formulation captures the requirements of the real stowage problem.

### When this tier is useful

This exact formulation is useful for small instances, for studying how constraints affect an assignment, and for obtaining an optimal benchmark when the model is an appropriate representation of the planning task. Larger applications require careful attention to computational limits and to the operational constraints represented in the model.